# Symmetric-projected ABBA guiding-center symplecticity study

This experiment evaluates the physical guiding-center map produced by `SymmetricProjectedABBA`, the second-order endpoint-time ABBA splitting closed by Hairer's symmetric projection. For a step $h$ with $s=h/2$, the duplicated state follows $A_{s,t_n}$, $B_{s,t_n}$, $B_{s,t_n+h}$, and $A_{s,t_n+h}$. The projection multiplier solves

$$R(\mu)=u^{(2)}(\mu)-v^{(2)}(\mu)+2\mu=0,$$

using the exact state-dependent residual Jacobian derived in `docs/tex/ABBA_implicit/ABBA_implicit.tex`.

For every complete finite-tolerance ABBA step, this study differentiates the emitted numerical map by centered finite differences. If $J_n$ is one step Jacobian and $DG_n$ is the accumulated Jacobian from the initial condition, the recorded defects are

$$\varepsilon_{\mathrm{local},n}=\frac{\|J_n^T\Omega J_n-\Omega\|_F}{\|\Omega\|_F},\qquad \varepsilon_{\mathrm{flow},n}=\frac{\|DG_n^T\Omega DG_n-\Omega\|_F}{\|\Omega\|_F}.$$

The transported polygon area, $|\det(DG_n)-1|$, Newton residuals, iteration counts, and projection multipliers provide complementary geometric and nonlinear-solver diagnostics.

API migration: this notebook uses the current simulation API. Physical rho and eta belong to dynamics or study settings; initial configurations store geometry. Stored outputs were cleared and should be regenerated before scientific interpretation.


In [ ]:
import numpy as np

from studies import (
    ABBASymplecticityConfig,
    RandomPotentialConfig,
    centered_circle,
    pi_area_steps,
    run_abba_symplecticity_study,
)
from visualization import display_animation

## Reproducible configuration

The potential, boundary geometry, physical parameters, integration interval, three ABBA steps, observation interval, finite-difference scale, and Newton stopping parameters are explicit below. The physical configuration and differentiation scale match the RK4 experiment in this directory so their defect magnitudes can be compared directly.

In [ ]:
potential_config = RandomPotentialConfig(
    amplitude=0.7,
    max_wave_number=25,
    nx=64,
    ny=64,
    seed=27,
    interpolation_order=5,
)
potential = potential_config.build()

circle_radius = 0.5
circle_points = 16
rho = 0.3
circle = centered_circle(
    potential,
    radius=circle_radius,
    points=circle_points,
    
)

finite_difference_relative_step = float(np.cbrt(np.finfo(float).eps))
study_config = ABBASymplecticityConfig(
    rho=rho,
    steps=pi_area_steps(40, 80, 160),
    t_span=(0.0, 4 * np.pi),
    save_interval=np.pi / 8,
    chunk_size=16,
    progress=True,
    block_prefix="symmetric_projected_abba_symplecticity",
    finite_difference_relative_step=finite_difference_relative_step,
    newton_absolute_tolerance=1e-13,
    newton_relative_tolerance=1e-12,
    newton_max_iterations=12,
)

print(potential_config)
print(study_config)
print(
    f"Circle: {circle_points} points; t={study_config.t_span}; "
    f"{study_config.output_sample_count} saved states; "
    f"finite-difference relative step={finite_difference_relative_step:.8e}"
)

## Projected ABBA integrations and persisted Jacobians

Newton starts from $\mu_0=0$ on every step, uses the infinity norm for its stopping test, and reevaluates the exact residual Jacobian after each correction. Every main-grid ABBA step contributes to the accumulated finite-difference Jacobian. Scalar diagnostics and full local and accumulated Jacobians are persisted at the common observation times.

In [ ]:
result = run_abba_symplecticity_study(
    potential,
    circle,
    notebook_path=(
        "notebooks/experiments/symplecticity/"
        "gc_area_and_symmetric_projected_abba_symplecticity.ipynb"
    ),
    config=study_config,
    metadata={
        **potential_config.metadata(),
        "circle_radius": circle_radius,
        "study_kind": "versioned_symplecticity_experiment",
    },
)
result.print_summary()

## Time-dependent diagnostics

The local matrix defect tests each projected ABBA step independently. The accumulated defect and determinant error test the complete discrete flow from the initial boundary, while the area panel shows the transported polygon geometry.

In [ ]:
diagnostic_figure, diagnostic_axes = result.plot_diagnostics()

## Measured symplecticity floor

An exact-root symmetric projection is structurally symplectic, but this experiment differentiates the finite-tolerance implementation. Once the local defect is controlled by finite-difference, Newton, and floating-point errors, no temporal convergence order should be fitted. The log–log plot therefore compares the measured floor across step sizes.

In [ ]:
defect_floor_figure, defect_floor_axis = result.plot_defect_floor()

## Nonlinear projection diagnostics

The iteration and residual histories verify that the projection solve remains below its configured stopping threshold. The maximum multiplier is expected to decrease approximately as $O(h^3)$; this is a consistency property of the symmetric projection, not the global trajectory order.

In [ ]:
solver_figure, solver_axes = result.plot_solver_diagnostics()

## Comparative animation

The animation synchronizes the effective potential, transported ABBA contours, relative polygon-area error, and accumulated physical-flow symplecticity defect. The same color identifies each ABBA step size in every panel.

In [ ]:
display_animation(
    result.animate(
        frames=None,
        interval=120,
    )
)

## Interpretation

The final interpretation is completed after executing the versioned experiment. It must distinguish the measured finite-difference floor from an exact symbolic proof, compare local and accumulated defects, verify the Newton solve and $O(h^3)$ multiplier scaling, and separate polygonal spatial error from temporal symplecticity.